# ML Basics · A good fit is not yet a useful prediction

SmartVal needs to estimate a listing's monthly rent from its size, distance from the centre, and age. You can fit a model and print a small error. The awkward question is: **small error on which listings?**

The [free-kick chapter](../01-math-foundations/math-foundations-for-ml.ipynb) supplied a rule from physics. Here we give the computer examples and ask it to fit a rule. A **feature** is a supplied input, such as size. A **target** is the answer we want to predict, such as rent. **Supervised learning** fits adjustable model values using paired features and targets.

We will generate 240 synthetic listings so the experiment is repeatable and needs no download. The noise and price rule are made up; this teaches evaluation, not a local property market.

| The question SmartVal asks | What we'll inspect |
| --- | --- |
| Did we learn from the same rows we later call “unseen”? | A 144/48/48 split with traceable row IDs |
| Are occasional huge misses acceptable? | The same errors under MAE and MSE |
| Which listings should trigger a high-rent flag? | One probability model at several thresholds |
| Does extra flexibility help on new examples? | Training and validation error across polynomial degrees |

The first cell creates `X`, a `(240, 3)` table: one listing per row, columns `[size_m2, distance_km, age_years]`. `y` has shape `(240,)`: one matching rent per row. `row_ids` lets us follow each listing through the split.

```text
one listing [size, distance, age] -> fitted rule -> predicted monthly rent
240 such rows X                 -> same rule  -> 240 predictions
```

The generating formula includes a curved size effect and random noise. We know that formula because this is a teaching dataset; the fitted models below receive the examples, not the formula.

In [ ]:
# -- Imports and one reproducible SmartVal dataset ----------------------------
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rng = np.random.default_rng(17)
n_listings = 240
size_m2 = rng.uniform(28, 145, n_listings)
distance_km = rng.uniform(0.3, 18.0, n_listings)
age_years = rng.uniform(0, 70, n_listings)
rent = 620 + 13.5 * size_m2 - 29 * distance_km - 4.5 * age_years
rent += 0.055 * (size_m2 - 82) ** 2 + rng.normal(0, 120, n_listings)
# Three (240,) columns -> (240, 3): keep each listing's features on the same row.
X = np.column_stack([size_m2, distance_km, age_years])
y = rent
row_ids = np.arange(n_listings)
print(f"Dataset contract: X={X.shape} (listing, feature), y={y.shape} (listing,)")


### Practise, choose, then report

Imagine fitting SmartVal to 144 listings and getting every rent right. That might mean it found a useful pattern, or merely remembered those listings. Reusing them to judge future accuracy cannot tell the difference.

Give separate rows separate jobs:

- **Training:** fit parameters, the adjustable coefficients and intercept.
- **Validation:** compare choices such as polynomial degree or flagging threshold.
- **Test:** report performance for a chosen model without using the result to keep changing it.

A **hyperparameter** is a setting you choose around fitting: Ridge's `alpha` or polynomial degree, for example. A model coefficient is a parameter it learns. Both can become indirectly tuned to a validation set if you repeatedly choose what scores well there.

> **Visual to add — Three piles with different jobs**
>
> Draw 240 numbered listing cards split into 144 practice cards, 48 choice cards, and 48 report cards. Route practice cards into fitting and choice cards into selecting a degree or threshold. Put the report pile aside until a model is chosen. Do not draw validation or test arrows into the fitted coefficients or scaler statistics; only the training pile teaches those values.
>
> **Proposed caption:** “A useful score needs examples that did not help choose the answer.”

## 1 · Split before learning the scale

Size is measured in square metres, distance in kilometres, and age in years. Their raw magnitudes are not directly comparable. A **scaler** subtracts a feature's training mean and divides by its training standard deviation. It changes units, not which listing a row describes.

Illustrative arithmetic, not statistics from this run: if training size has mean `80` and standard deviation `20`, a `100 m²` listing becomes `(100-80)/20 = 1`. It is one training standard deviation above average, not “1 square metre.”

**Why split first?** Even the averages and spreads must come from training rows. With illustrative training sizes `40, 60`, the mean is `50`; adding a validation listing of size `140` would change it to `80`. You would have let supposedly unseen data help prepare the model.

The next cell splits features, rents, and IDs together: first 144 training rows and 96 held-out rows, then 48 validation and 48 test rows. Splitting columns independently would scramble which rent belongs to which home.

The overlap check verifies that no ID occurs in two piles. It does not prove the absence of duplicates, future information in features, or time-related leakage. A random split suits these synthetic independent rows; future real rents may need a chronological split.

In [ ]:
# -- Create disjoint train, validation, and test contracts --------------------
# Split rows together: (240, 3) -> (144, 3) train + (96, 3) holdout;
# matching (240,) targets/IDs follow those rows so labels cannot drift.
X_train, X_holdout, y_train, y_holdout, id_train, id_holdout = train_test_split(
    X, y, row_ids, test_size=0.40, random_state=17
)
# (96, 3) -> two (48, 3) piles; each keeps its own (48,) targets and IDs.
X_val, X_test, y_val, y_test, id_val, id_test = train_test_split(
    X_holdout, y_holdout, id_holdout, test_size=0.50, random_state=17
)

overlap = (
    set(id_train) & set(id_val)
    or set(id_train) & set(id_test)
    or set(id_val) & set(id_test)
)
print(f"Rows: train={len(id_train)}, validation={len(id_val)}, test={len(id_test)}")
print(f"Isolation check: overlap={len(overlap)}; zero means the evidence roles are separate.")


### Now turn a listing into a guess

The next model learns a weighted sum of scaled features plus an intercept. A coefficient says how the prediction changes when one scaled input increases by one and the others stay fixed.

For illustration only, suppose the scaled row is `[1, -1, 0]`, the weights are `[200, -80, -40]`, and the intercept is `1500`:

```text
1500 + 200*1 + (-80)*(-1) + (-40)*0 = 1780 rent units
```

Those are not the fitted weights below. They show the calculation that `predict` will perform with its own learned values. The intercept is the prediction at the all-zero scaled row, not a universal base rent.

Compare a guess with its target to get an **error**: prediction minus true rent. A prediction of `1780` against `1700` has error `+80`; `1620` against `1700` has error `-80`. Averaging signed errors would let those misses cancel. We need a score that still notices both.

## 2 · One bad miss can dominate a score

**Mean absolute error (MAE)** averages error sizes. **Mean squared error (MSE)** squares them first, so large misses get much more weight. MAE stays in rent units; MSE is in rent-units squared.

The pipeline below first learns `StandardScaler` statistics from training rows, then fits `Ridge(alpha=2.0)`. Ridge seeks a squared-error fit while penalizing large coefficients. That penalty is **regularization**: it makes a large response to a feature cost something. It does not guarantee a better held-out model.

`fit` learns the scaler and coefficients. `predict` reuses both without learning from validation or test rows. This library call does not expose a gradient-descent loop; fitting need not use one.

We report validation MSE and MAE for the same predictions. The cell also prints a test MAE for this **fixed Ridge baseline**. The later degree experiment is separate: it does not select, refit, and test a final polynomial winner. If you use the test result to revise the baseline, it is no longer untouched evidence for that revision.

> **Visual to add — Four misses, two ways to count them**
>
> Draw four error bars labelled 40, 40, 40, and 400 rent units. Under the first row, show their absolute contributions; under the second, squares with areas 1600, 1600, 1600, and 160000. Highlight the large miss's roughly 77% versus 97% share. Do not use those shares as gradient arrows: each weight's gradient also depends on the input features.
>
> **Proposed caption:** “Squaring changes which mistakes dominate, not just the units of the score.”

The following four-error experiment gives `MAE = 130` and `MSE = 41,200`. It measures contributions to the scores, not an actual parameter update. A **loss** is what fitting tries to reduce; a **metric** is what you report for a question you care about. The same calculation can serve both jobs.

In [ ]:
# -- Fit one compact regression implementation and report two contracts -------
# Scaling keeps (rows, 3): reuse three train-only feature statistics across rows.
regressor = make_pipeline(StandardScaler(), Ridge(alpha=2.0))
regressor.fit(X_train, y_train)

val_prediction = regressor.predict(X_val)
test_prediction = regressor.predict(X_test)
val_mse = mean_squared_error(y_val, val_prediction)
val_mae = mean_absolute_error(y_val, val_prediction)
test_mae = mean_absolute_error(y_test, test_prediction)

print(f"Validation MSE: {val_mse:,.0f} rent-units squared")
print(f"Validation MAE: {val_mae:,.1f} rent units")
print(f"Final untouched test MAE: {test_mae:,.1f} rent units")


In [ ]:
# -- Measure why squared loss reacts more strongly to one large miss ----------
errors = np.array([40.0, 40.0, 40.0, 400.0])
mae_contribution = np.abs(errors)
mse_contribution = errors**2
# Take the last scalar from each (4,) vector to compare the same outlier under both scores.
large_share_mae = mae_contribution[-1] / mae_contribution.sum()
large_share_mse = mse_contribution[-1] / mse_contribution.sum()
print(f"Large miss share under MAE: {large_share_mae:.1%}")
print(f"Large miss share under MSE: {large_share_mse:.1%}")
print("Measured consequence: squaring lets the largest miss dominate the update signal.")


### A rent estimate is a number. A flag is an action.

Predicting rent is **regression**. Deciding whether a listing belongs to a high-rent group is **classification**. “Logistic regression” is confusingly named: it is the classifier we use below, not another continuous-rent predictor.

For this exercise, “high rent” means at least the **training-set median** rent. The median is the middle training value; the same cutoff labels validation rows. This is an exercise-defined category, not an externally verified property valuation.

Logistic regression mixes the scaled features into a score and applies a **sigmoid**, a smooth conversion to a number between zero and one. We use that as an estimated probability of class 1. The output is not a certainty, and the notebook does not test calibration: whether predictions of 0.7 are right about 70% of the time.

## 3 · The model gives chances. You choose when to act.

Suppose three validation listings have predicted high-rent probabilities `[0.40, 0.65, 0.80]` and actual labels `[0, 1, 1]`. This is an illustrative three-row example, not the run's predictions.

- At threshold `0.35`, all three are flagged: two correct flags out of three, and both true positives found.
- At threshold `0.70`, only the last is flagged: one correct flag out of one, but only one of two true positives found.

**Precision** asks “among our flags, what fraction were right?” **Recall** asks “among listings that qualify, what fraction did we find?” The example moves precision from `2/3` to `1` while recall moves from `1` to `1/2`.

That is why `0.5` is not a law. Missing an important listing and bothering someone with an unnecessary flag have different costs. Raising a threshold removes flags; precision need not improve at every step on a finite dataset.

> **Visual to add — The same shortlist, a moving cutoff**
>
> Place the three listing cards at probabilities 0.40, 0.65, and 0.80 on one ruler, with true labels 0, 1, and 1 visible underneath. Compare cutoffs 0.35 and 0.70 using the exact flagged cards and counts above. Do not move a card when the cutoff changes or draw feedback into the fitted weights: only the action rule changed.
>
> **Proposed caption:** “A stricter cutoff changes who gets flagged without retraining the model.”

In the code, `predict_proba` returns `(48, 2)`: one row per listing, columns for classes 0 and 1. `[:, 1]` keeps only the class-1 column, giving `(48,)`. Comparing that vector with a threshold creates 48 decisions. The three printed precision/recall pairs reuse exactly those probabilities.

In [ ]:
# -- Separate probability estimation from the threshold decision -------------
high_value_cutoff = np.median(y_train)
y_train_class = (y_train >= high_value_cutoff).astype(int)
y_val_class = (y_val >= high_value_cutoff).astype(int)
classifier = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500))
classifier.fit(X_train, y_train_class)
# (48, 2) -> (48,): take the high-value class column, one probability per listing.
val_probability = classifier.predict_proba(X_val)[:, 1]

for threshold in (0.35, 0.50, 0.70):
    decision = (val_probability >= threshold).astype(int)
    precision = precision_score(y_val_class, decision, zero_division=0)
    recall = recall_score(y_val_class, decision, zero_division=0)
    print(f"threshold={threshold:.2f} -> precision={precision:.2f}, recall={recall:.2f}")
print("Measured consequence: the same probabilities support different operating decisions.")


### The test is fair. Can the model express the pattern?

Our data-generating rent rule includes a bend with size. A straight size-only trend cannot express it. Giving the model size squared lets it bend; adding still higher powers allows more complicated curves. That freedom is **capacity**.

For an illustrative size value `2`, degree 1 supplies `[2]`, degree 2 supplies `[2, 4]`, and degree 5 supplies `[2, 4, 8, 16, 32]`. These are different views of the same listing, not extra listings or new observations.

The next experiment deliberately uses **size alone** to isolate degree. It drops distance and age, so its errors are not a clean comparison with the earlier three-feature Ridge model.

## 4 · More freedom can fit noise too

The loop changes degree across `1, 2, 5, 10`. For each degree it builds the power columns, scales them using training rows, fits ordinary linear regression, and reports training and validation MAE. “Linear” here means linear in the fitted coefficients; powers of size still produce a curved relationship with size.

If a model misses a useful broad pattern, it **underfits**. If extra detail improves practice results but worsens new examples, it **overfits**. Neither is diagnosed from the training score alone.

One subtlety: these regressions minimize **squared error**, but the table reports **absolute error**. With nested feature sets, the best training squared error cannot worsen in exact arithmetic. Training MAE has no matching monotonic guarantee, and high-degree numerical problems can complicate the fit. Read the measured curves rather than expecting a scripted U shape.

`best_degree` selects the smallest **validation MAE among these four trials**. The test rows are not consulted for that choice. There is no final polynomial test evaluation below; choosing well on validation is not the same as reporting new test evidence.

Regularization was used in the earlier Ridge fit, not in this unpenalized degree sweep. Degree limits the menu of features; a coefficient penalty limits how strongly a model uses its available features. Different controls, related complaint.

In [ ]:
# -- Compare capacity with one changed variable: polynomial degree ------------
train_mae_by_degree = []
val_mae_by_degree = []
for degree in (1, 2, 5, 10):
    model = make_pipeline(
        # (rows, 1) -> (rows, degree): give the size feature more powers, not more listings.
        PolynomialFeatures(degree=degree, include_bias=False),
        StandardScaler(),
        LinearRegression(),
    )
    # Keep size only: (144, 3)/(48, 3) -> (144, 1)/(48, 1).
    # :1 keeps the feature axis that scikit-learn expects; [:, 0] would drop it.
    one_feature_train = X_train[:, :1]
    one_feature_val = X_val[:, :1]
    model.fit(one_feature_train, y_train)
    train_mae = mean_absolute_error(y_train, model.predict(one_feature_train))
    validation_mae = mean_absolute_error(y_val, model.predict(one_feature_val))
    train_mae_by_degree.append(train_mae)
    val_mae_by_degree.append(validation_mae)
    print(f"degree={degree:2d}: train MAE={train_mae:6.1f}, validation MAE={validation_mae:6.1f}")

best_degree = (1, 2, 5, 10)[int(np.argmin(val_mae_by_degree))]
print(f"Validation chooses degree {best_degree}; test data was not consulted.")


In [ ]:
# -- Visual evidence: training fit and validation evidence on one figure -------
degrees = np.array([1, 2, 5, 10])
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(degrees, train_mae_by_degree, marker="o", label="train MAE")
ax.plot(degrees, val_mae_by_degree, marker="o", label="validation MAE")
ax.set(xlabel="polynomial degree", ylabel="MAE", title="Capacity changes two error curves")
ax.legend()
ax.grid(alpha=0.3)
plt.show()
print("Read the gap, not just the lowest training point.")


### Your turn · Change one action rule

Keep the fitted probabilities fixed. Before running the next cell, choose whether increasing `YOUR_THRESHOLD` from `0.60` should create more flags or fewer.

Then compare its validation precision and recall with the earlier thresholds. The range assertion only checks that the cutoff is legal; it does not decide whether your trade-off is useful. No retraining is needed to answer this question.

In [ ]:
# -- CHANGE THIS: choose one threshold, then inspect the measured trade-off ----
YOUR_THRESHOLD = 0.60
your_decision = (val_probability >= YOUR_THRESHOLD).astype(int)
your_precision = precision_score(y_val_class, your_decision, zero_division=0)
your_recall = recall_score(y_val_class, your_decision, zero_division=0)
assert 0.0 <= YOUR_THRESHOLD <= 1.0
print(f"At {YOUR_THRESHOLD:.2f}: precision={your_precision:.2f}, recall={your_recall:.2f}")
print("The correctness check is the explicit threshold and the held-out measurement.")


## The short version

- A row has supplied features and a target. Fit on some rows; judge on different ones.
- Split before fitting preprocessing as well as before fitting coefficients.
- MAE counts typical miss size. MSE makes a few large misses much more influential.
- A model probability and a thresholded action are different outputs. Thresholds trade missed positives against unnecessary flags.
- More capacity can capture a real bend or chase noise. Validation helps distinguish them.
- The printed test score belongs to the fixed Ridge baseline, not to a later validation-selected polynomial.

**Where we stopped**

- **Built and ran:** row isolation checks, scaled Ridge regression, MAE/MSE contributions, a fixed classifier at several thresholds, and a size-only degree comparison.
- **Talked through:** parameters versus hyperparameters, regularization, leakage, underfitting, overfitting, and calibration.
- **Left for later:** repeated cross-validation, time-aware evaluation on real data, class-imbalance remedies, and explicit optimizer loops.

Keep an eye on mismatched row order, preprocessing fitted on all rows, and decisions repeatedly tuned against the test score. A neatly separated ID list cannot catch every leak.

Fair evaluation can reveal a bad rule. It cannot give a straight rule a shape it does not have. [SmartVal's neural-network chapter](../03-neural-networks-and-backprop/01-smartval-neural-networks-and-backprop.ipynb) keeps that distinction and switches to four two-signal cases, so you can see exactly why hidden features become necessary.